# Progressive Identity Verification Under Multimodal Evidence

A synthetic experimental notebook for studying how confidence in a continuing identity changes as evidence accumulates across **time**, **modalities**, and **cross-modal relationships**.

The notebook does **not** use real biometric data and is not designed for deployment as a real-world identification or surveillance system. Instead, it creates synthetic latent identities and simulated observations corresponding loosely to modalities such as voice, face, gait, body geometry, physiology, anatomy, and temporal history.

The central question is:

> How does the admissible identity set contract as independent and mutually consistent evidence accumulates?

The experiments focus on six ideas:

1. More observations within one modality usually improve discrimination, but with diminishing returns.
2. New modalities can reduce ambiguity more than additional samples of an already-saturated modality.
3. Cross-modal couplings can be more discriminative than either modality alone.
4. Temporal continuity can distinguish a plausible identity trajectory from disconnected snapshots.
5. Better imitation and stronger verification can improve simultaneously.
6. Identity confidence is best treated as accumulated evidence, not as a single binary biometric match.

In [ ]:
from __future__ import annotations

import math
import secrets
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(20260824)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 100)

## Synthetic identity model

Each synthetic person has a latent state containing several modality-specific feature vectors plus a small number of cross-modal coupling parameters.

These are abstractions only. For example, the `voice` vector does not represent actual speech features, and `physiology` does not represent real medical measurements. They simply let us test how independent evidence channels combine.

In [ ]:
MODALITIES = {
    "voice": 10,
    "face": 12,
    "gait": 8,
    "geometry": 8,
    "physiology": 6,
    "anatomy": 6,
}

@dataclass
class SyntheticIdentity:
    identity_id: int
    latent: dict
    coupling_voice_face: np.ndarray
    coupling_gait_geometry: np.ndarray
    aging_drift: dict

def make_identity(identity_id: int) -> SyntheticIdentity:
    latent = {
        name: rng.normal(0, 1, size=d)
        for name, d in MODALITIES.items()
    }
    coupling_voice_face = rng.normal(0, 0.35, size=4)
    coupling_gait_geometry = rng.normal(0, 0.35, size=4)
    aging_drift = {
        name: rng.normal(0, 0.03, size=d)
        for name, d in MODALITIES.items()
    }
    return SyntheticIdentity(
        identity_id=identity_id,
        latent=latent,
        coupling_voice_face=coupling_voice_face,
        coupling_gait_geometry=coupling_gait_geometry,
        aging_drift=aging_drift,
    )

N_IDENTITIES = 1000
IDENTITIES = [make_identity(i) for i in range(N_IDENTITIES)]

## Observation model

An observation is a noisy view of one modality at a specified time. More exposure reduces noise approximately with the square root of duration, while an irreducible noise floor prevents perfect certainty.

This captures the rough intuition that 30,000 hours of evidence can be much more informative than 15 seconds, but that repeated observations eventually saturate.

In [ ]:
BASE_NOISE = {
    "voice": 1.30,
    "face": 1.00,
    "gait": 1.15,
    "geometry": 0.85,
    "physiology": 0.95,
    "anatomy": 0.75,
}

NOISE_FLOOR = {
    "voice": 0.18,
    "face": 0.16,
    "gait": 0.22,
    "geometry": 0.14,
    "physiology": 0.20,
    "anatomy": 0.12,
}

def duration_noise(modality, duration_seconds):
    exposure = max(duration_seconds, 1e-6)
    return NOISE_FLOOR[modality] + BASE_NOISE[modality] / math.sqrt(exposure + 1.0)

def latent_at_time(identity, modality, years):
    return identity.latent[modality] + years * identity.aging_drift[modality]

def observe(identity, modality, duration_seconds=1.0, years=0.0):
    sigma = duration_noise(modality, duration_seconds)
    truth = latent_at_time(identity, modality, years)
    return truth + rng.normal(0, sigma, size=truth.shape)

def squared_distance(a, b):
    return float(np.sum((a - b) ** 2))

## Verification score

For a candidate identity, we score how compatible the observed evidence is with that candidate's expected latent state. Lower distance means stronger compatibility.

The score below acts like a simplified negative log-likelihood.

In [ ]:
def modality_nll(candidate, modality, observation, duration_seconds=1.0, years=0.0):
    sigma = duration_noise(modality, duration_seconds)
    expected = latent_at_time(candidate, modality, years)
    return squared_distance(observation, expected) / (2 * sigma**2)

def combined_nll(candidate, evidence):
    total = 0.0
    for item in evidence:
        total += modality_nll(
            candidate,
            item["modality"],
            item["observation"],
            item["duration_seconds"],
            item["years"],
        )
    return total

def rank_candidates(evidence, identities=IDENTITIES):
    rows = []
    for candidate in identities:
        rows.append((candidate.identity_id, combined_nll(candidate, evidence)))
    rows.sort(key=lambda x: x[1])
    return rows

def true_rank(evidence, true_id):
    ranked = rank_candidates(evidence)
    for rank, (identity_id, score) in enumerate(ranked, start=1):
        if identity_id == true_id:
            return rank, score
    raise RuntimeError("true identity missing")

# Experiment 1 — Voice duration and diminishing returns

This experiment asks how the true identity's rank changes as voice evidence grows from milliseconds to tens of thousands of hours.

In [ ]:
def experiment_01_voice_duration(true_id=7):
    identity = IDENTITIES[true_id]
    durations = [
        0.001,
        0.01,
        0.1,
        1,
        15,
        60,
        300,
        3600,
        10 * 3600,
        100 * 3600,
        1000 * 3600,
        30000 * 3600,
    ]

    rows = []
    for duration in durations:
        obs = observe(identity, "voice", duration_seconds=duration)
        evidence = [{
            "modality": "voice",
            "observation": obs,
            "duration_seconds": duration,
            "years": 0.0,
        }]
        rank, score = true_rank(evidence, true_id)
        rows.append({
            "duration_seconds": duration,
            "duration_hours": duration / 3600,
            "true_rank": rank,
            "true_score": score,
        })
    return pd.DataFrame(rows)

voice_duration_df = experiment_01_voice_duration()
voice_duration_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(voice_duration_df["duration_seconds"], voice_duration_df["true_rank"], marker="o")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Voice exposure (seconds, log scale)")
plt.ylabel("True identity rank (log scale)")
plt.title("Experiment 1: Identity rank vs voice duration")
plt.grid(True, alpha=0.25)
plt.show()

# Experiment 2 — Repeated samples versus one long sample

A system may receive ten 30-second voice samples under varied conditions or one continuous 300-second sample. This experiment compares accumulation across repeated observations with a single observation of equal total duration.

In [ ]:
def experiment_02_repetition(true_id=11, n_reps=100):
    identity = IDENTITIES[true_id]
    rows = []

    for _ in range(n_reps):
        repeated = []
        for _ in range(10):
            repeated.append({
                "modality": "voice",
                "observation": observe(identity, "voice", 30),
                "duration_seconds": 30,
                "years": 0.0,
            })

        single = [{
            "modality": "voice",
            "observation": observe(identity, "voice", 300),
            "duration_seconds": 300,
            "years": 0.0,
        }]

        rep_rank, _ = true_rank(repeated, true_id)
        single_rank, _ = true_rank(single, true_id)

        rows.append({
            "repeated_rank": rep_rank,
            "single_rank": single_rank,
        })

    return pd.DataFrame(rows)

repeat_df = experiment_02_repetition()
repeat_df.describe()

# Experiment 3 — Breadth versus depth

Compare one very deep voice archive against a moderate voice archive plus several additional modalities.

In [ ]:
def experiment_03_breadth_vs_depth(true_id=21, n_reps=100):
    identity = IDENTITIES[true_id]
    rows = []

    for _ in range(n_reps):
        deep_voice = [{
            "modality": "voice",
            "observation": observe(identity, "voice", 10000 * 3600),
            "duration_seconds": 10000 * 3600,
            "years": 0.0,
        }]

        multimodal = []
        durations = {
            "voice": 3600,
            "face": 3600,
            "gait": 600,
            "geometry": 60,
            "physiology": 30,
            "anatomy": 30,
        }
        for modality, dur in durations.items():
            multimodal.append({
                "modality": modality,
                "observation": observe(identity, modality, dur),
                "duration_seconds": dur,
                "years": 0.0,
            })

        deep_rank, _ = true_rank(deep_voice, true_id)
        multi_rank, _ = true_rank(multimodal, true_id)

        rows.append({
            "deep_voice_rank": deep_rank,
            "multimodal_rank": multi_rank,
        })

    return pd.DataFrame(rows)

breadth_depth_df = experiment_03_breadth_vs_depth()
breadth_depth_df.describe()

# Experiment 4 — Sequential contraction of the admissible set

Instead of only measuring rank, this experiment counts how many candidate identities remain within a compatibility margin as evidence is added one modality at a time.

In [ ]:
def admissible_count(evidence, margin=12.0):
    ranked = rank_candidates(evidence)
    best_score = ranked[0][1]
    return sum(score <= best_score + margin for _, score in ranked)

def experiment_04_admissible_contraction(true_id=33):
    identity = IDENTITIES[true_id]
    sequence = [
        ("voice", 15),
        ("voice", 3600),
        ("face", 600),
        ("gait", 300),
        ("geometry", 30),
        ("physiology", 20),
        ("anatomy", 20),
    ]

    evidence = []
    rows = []

    for step, (modality, duration) in enumerate(sequence, start=1):
        evidence.append({
            "modality": modality,
            "observation": observe(identity, modality, duration),
            "duration_seconds": duration,
            "years": 0.0,
        })
        rank, _ = true_rank(evidence, true_id)
        rows.append({
            "step": step,
            "added_modality": modality,
            "duration_seconds": duration,
            "true_rank": rank,
            "admissible_candidates": admissible_count(evidence),
        })

    return pd.DataFrame(rows)

contraction_df = experiment_04_admissible_contraction()
contraction_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(contraction_df["step"], contraction_df["admissible_candidates"], marker="o")
plt.xlabel("Evidence accumulation step")
plt.ylabel("Admissible candidate identities")
plt.title("Experiment 4: Contraction of the admissible identity set")
plt.grid(True, alpha=0.25)
plt.show()

# Experiment 5 — Cross-modal coupling

Two candidates might independently match voice and face reasonably well, yet differ in the relationship between those modalities. This experiment adds a synthetic coupling signature and tests whether it improves discrimination.

In [ ]:
def coupling_observation(identity, duration_seconds=60):
    sigma = 0.5 / math.sqrt(duration_seconds + 1) + 0.08
    truth = identity.coupling_voice_face
    return truth + rng.normal(0, sigma, size=truth.shape)

def coupling_nll(candidate, observation, duration_seconds=60):
    sigma = 0.5 / math.sqrt(duration_seconds + 1) + 0.08
    return squared_distance(observation, candidate.coupling_voice_face) / (2 * sigma**2)

def rank_with_coupling(evidence, coupling_obs=None, coupling_duration=60):
    rows = []
    for candidate in IDENTITIES:
        score = combined_nll(candidate, evidence)
        if coupling_obs is not None:
            score += coupling_nll(candidate, coupling_obs, coupling_duration)
        rows.append((candidate.identity_id, score))
    rows.sort(key=lambda x: x[1])
    return rows

def rank_of(ranked, true_id):
    for i, (identity_id, score) in enumerate(ranked, start=1):
        if identity_id == true_id:
            return i

def experiment_05_cross_modal(true_id=44, n_reps=100):
    identity = IDENTITIES[true_id]
    rows = []

    for _ in range(n_reps):
        evidence = [
            {
                "modality": "voice",
                "observation": observe(identity, "voice", 120),
                "duration_seconds": 120,
                "years": 0.0,
            },
            {
                "modality": "face",
                "observation": observe(identity, "face", 120),
                "duration_seconds": 120,
                "years": 0.0,
            },
        ]
        coupling_obs = coupling_observation(identity, 120)

        without = rank_candidates(evidence)
        with_c = rank_with_coupling(evidence, coupling_obs, 120)

        rows.append({
            "rank_without_coupling": rank_of(without, true_id),
            "rank_with_coupling": rank_of(with_c, true_id),
        })

    return pd.DataFrame(rows)

coupling_df = experiment_05_cross_modal()
coupling_df.describe()

# Experiment 6 — Imitator improves with more voice data

This experiment introduces a synthetic imitator whose voice approximation improves as training exposure increases.

The verifier first uses voice only, then adds modalities the imitator does not model. This demonstrates the asymmetry between becoming a better imitation in one channel and becoming a globally consistent identity.

In [ ]:
def imitation_voice(true_identity, training_hours):
    # Approximation error shrinks with training exposure.
    sigma = 1.6 / math.sqrt(training_hours + 0.1) + 0.05
    return true_identity.latent["voice"] + rng.normal(0, sigma, size=MODALITIES["voice"])

def verification_score_against_reference(reference_identity, claimed_latent, modality, duration):
    sigma = duration_noise(modality, duration)
    expected = reference_identity.latent[modality]
    return squared_distance(claimed_latent, expected) / (2 * sigma**2)

def experiment_06_imitation(true_id=52):
    true_identity = IDENTITIES[true_id]
    training_hours = [0.001, 0.01, 0.1, 1, 5, 50, 500, 5000, 30000]
    rows = []

    for hours in training_hours:
        fake_voice = imitation_voice(true_identity, hours)

        voice_score = verification_score_against_reference(
            true_identity, fake_voice, "voice", duration=300
        )

        # Imitator has no special fit to the other channels.
        fake_face = rng.normal(0, 1, MODALITIES["face"])
        fake_gait = rng.normal(0, 1, MODALITIES["gait"])

        multi_score = (
            voice_score
            + verification_score_against_reference(true_identity, fake_face, "face", 300)
            + verification_score_against_reference(true_identity, fake_gait, "gait", 300)
        )

        rows.append({
            "training_hours": hours,
            "voice_only_score": voice_score,
            "multimodal_score": multi_score,
        })

    return pd.DataFrame(rows)

imitation_df = experiment_06_imitation()
imitation_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(imitation_df["training_hours"], imitation_df["voice_only_score"], marker="o", label="Voice-only verification")
plt.plot(imitation_df["training_hours"], imitation_df["multimodal_score"], marker="o", label="Multimodal verification")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Imitator voice training exposure (hours)")
plt.ylabel("Mismatch score (lower is better)")
plt.title("Experiment 6: Better imitation does not imply full identity")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()

# Experiment 7 — Temporal continuity

Identity is modeled here as a trajectory rather than a frozen vector. We compare a candidate whose observations follow the correct long-term drift against disconnected snapshots drawn from unrelated states.

In [ ]:
def trajectory_evidence(identity, modality, years_list, duration=300):
    return [
        {
            "modality": modality,
            "observation": observe(identity, modality, duration, years=year),
            "duration_seconds": duration,
            "years": year,
        }
        for year in years_list
    ]

def experiment_07_temporal_continuity(true_id=60):
    identity = IDENTITIES[true_id]
    years = [0, 5, 10, 20, 30]

    coherent = trajectory_evidence(identity, "face", years, duration=300)
    coherent_rank, coherent_score = true_rank(coherent, true_id)

    # Build a deliberately incoherent "history" from different identities.
    incoherent = []
    donor_ids = [71, 72, 73, 74, 75]
    for donor_id, year in zip(donor_ids, years):
        donor = IDENTITIES[donor_id]
        incoherent.append({
            "modality": "face",
            "observation": observe(donor, "face", 300, years=year),
            "duration_seconds": 300,
            "years": year,
        })

    incoherent_true_rank, incoherent_true_score = true_rank(incoherent, true_id)

    return pd.DataFrame([{
        "coherent_true_rank": coherent_rank,
        "coherent_true_score": coherent_score,
        "incoherent_true_rank": incoherent_true_rank,
        "incoherent_true_score": incoherent_true_score,
    }])

temporal_df = experiment_07_temporal_continuity()
temporal_df

# Experiment 8 — Same amount of data, different modality diversity

This holds the nominal evidence budget roughly constant while redistributing it between one modality and several modalities.

In [ ]:
def experiment_08_budget_allocation(true_id=81, n_reps=100):
    identity = IDENTITIES[true_id]
    rows = []

    for _ in range(n_reps):
        # Budget A: all concentrated in voice.
        a = [{
            "modality": "voice",
            "observation": observe(identity, "voice", 3600),
            "duration_seconds": 3600,
            "years": 0.0,
        }]

        # Budget B: distributed across four channels.
        b = []
        for modality, duration in [
            ("voice", 900),
            ("face", 900),
            ("gait", 600),
            ("geometry", 600),
        ]:
            b.append({
                "modality": modality,
                "observation": observe(identity, modality, duration),
                "duration_seconds": duration,
                "years": 0.0,
            })

        rank_a, _ = true_rank(a, true_id)
        rank_b, _ = true_rank(b, true_id)

        rows.append({
            "concentrated_rank": rank_a,
            "distributed_rank": rank_b,
        })

    return pd.DataFrame(rows)

budget_df = experiment_08_budget_allocation()
budget_df.describe()

# Experiment 9 — Correlated evidence and false confidence

Ten measurements are not equivalent to ten independent measurements if they are strongly correlated. This experiment compares nominally equal evidence counts under independent and correlated noise.

In [ ]:
def correlated_observations(identity, modality, n, duration, correlation=0.95):
    sigma = duration_noise(modality, duration)
    truth = identity.latent[modality]

    shared = rng.normal(0, sigma, size=truth.shape)
    observations = []
    for _ in range(n):
        independent = rng.normal(0, sigma, size=truth.shape)
        noise = math.sqrt(correlation) * shared + math.sqrt(1 - correlation) * independent
        observations.append(truth + noise)
    return observations

def experiment_09_correlation(true_id=93, n_trials=100):
    identity = IDENTITIES[true_id]
    rows = []

    for _ in range(n_trials):
        independent_evidence = []
        for _ in range(10):
            independent_evidence.append({
                "modality": "voice",
                "observation": observe(identity, "voice", 30),
                "duration_seconds": 30,
                "years": 0.0,
            })

        correlated = correlated_observations(identity, "voice", 10, 30, correlation=0.95)
        correlated_evidence = [
            {
                "modality": "voice",
                "observation": obs,
                "duration_seconds": 30,
                "years": 0.0,
            }
            for obs in correlated
        ]

        ind_rank, _ = true_rank(independent_evidence, true_id)
        corr_rank, _ = true_rank(correlated_evidence, true_id)

        rows.append({
            "independent_rank": ind_rank,
            "correlated_rank": corr_rank,
        })

    return pd.DataFrame(rows)

correlation_df = experiment_09_correlation()
correlation_df.describe()

# Experiment 10 — Conflicting modalities

What happens when strong evidence from one channel supports one identity while another modality supports a different identity?

This models contradictory evidence and shows why a verifier should expose conflict rather than collapse everything into a single opaque score.

In [ ]:
def experiment_10_conflict(id_a=101, id_b=202):
    a = IDENTITIES[id_a]
    b = IDENTITIES[id_b]

    evidence = [
        {
            "modality": "voice",
            "observation": observe(a, "voice", 3600),
            "duration_seconds": 3600,
            "years": 0.0,
        },
        {
            "modality": "face",
            "observation": observe(b, "face", 3600),
            "duration_seconds": 3600,
            "years": 0.0,
        },
    ]

    ranked = rank_candidates(evidence)[:10]

    modality_breakdown = []
    for candidate_id in [id_a, id_b]:
        candidate = IDENTITIES[candidate_id]
        voice_nll = modality_nll(candidate, "voice", evidence[0]["observation"], 3600, 0)
        face_nll = modality_nll(candidate, "face", evidence[1]["observation"], 3600, 0)
        modality_breakdown.append({
            "candidate_id": candidate_id,
            "voice_nll": voice_nll,
            "face_nll": face_nll,
            "combined_nll": voice_nll + face_nll,
        })

    return pd.DataFrame(modality_breakdown), pd.DataFrame(ranked, columns=["candidate_id", "combined_nll"])

conflict_breakdown_df, conflict_top_df = experiment_10_conflict()
conflict_breakdown_df

# Experiment 11 — Evidence age and stale identity models

A verifier may have abundant historical evidence yet poor knowledge of the current state. This experiment compares matching with and without an aging model.

In [ ]:
def stale_modality_nll(candidate, modality, observation, duration_seconds):
    sigma = duration_noise(modality, duration_seconds)
    expected = candidate.latent[modality]  # ignores temporal drift
    return squared_distance(observation, expected) / (2 * sigma**2)

def rank_candidates_stale(evidence):
    rows = []
    for candidate in IDENTITIES:
        score = 0.0
        for item in evidence:
            score += stale_modality_nll(
                candidate,
                item["modality"],
                item["observation"],
                item["duration_seconds"],
            )
        rows.append((candidate.identity_id, score))
    rows.sort(key=lambda x: x[1])
    return rows

def experiment_11_staleness(true_id=304):
    identity = IDENTITIES[true_id]
    years = 35

    evidence = [{
        "modality": "face",
        "observation": observe(identity, "face", 600, years=years),
        "duration_seconds": 600,
        "years": years,
    }]

    temporal_rank = rank_of(rank_candidates(evidence), true_id)
    stale_rank = rank_of(rank_candidates_stale(evidence), true_id)

    return pd.DataFrame([{
        "years_since_reference": years,
        "rank_with_temporal_model": temporal_rank,
        "rank_with_stale_model": stale_rank,
    }])

staleness_df = experiment_11_staleness()
staleness_df

# Experiment 12 — Stronger verifier and stronger imitator

This final experiment models an arms race. The imitator improves in the voice channel as training exposure grows, while the verifier progressively adds independent modalities and couplings.

The question is not whether imitation improves—it does—but whether the **verification boundary** can improve faster by expanding the set of constraints.

In [ ]:
def experiment_12_arms_race(true_id=411):
    identity = IDENTITIES[true_id]
    training_hours = np.logspace(-3, math.log10(30000), 20)

    rows = []
    for hours in training_hours:
        fake_voice = imitation_voice(identity, hours)

        voice_score = verification_score_against_reference(
            identity, fake_voice, "voice", 300
        )

        # Independent synthetic impostor channels.
        fake_face = rng.normal(0, 1, MODALITIES["face"])
        fake_gait = rng.normal(0, 1, MODALITIES["gait"])
        fake_geometry = rng.normal(0, 1, MODALITIES["geometry"])
        fake_coupling = rng.normal(0, 1, 4)

        level1 = voice_score

        level2 = (
            level1
            + verification_score_against_reference(identity, fake_face, "face", 300)
        )

        level3 = (
            level2
            + verification_score_against_reference(identity, fake_gait, "gait", 300)
            + verification_score_against_reference(identity, fake_geometry, "geometry", 60)
        )

        coupling_sigma = 0.5 / math.sqrt(120 + 1) + 0.08
        coupling_score = squared_distance(
            fake_coupling, identity.coupling_voice_face
        ) / (2 * coupling_sigma**2)

        level4 = level3 + coupling_score

        rows.append({
            "training_hours": hours,
            "voice_only": level1,
            "voice_plus_face": level2,
            "multimodal": level3,
            "multimodal_plus_coupling": level4,
        })

    return pd.DataFrame(rows)

arms_race_df = experiment_12_arms_race()
arms_race_df.head()

In [ ]:
plt.figure(figsize=(9, 6))
plt.plot(arms_race_df["training_hours"], arms_race_df["voice_only"], label="Voice only")
plt.plot(arms_race_df["training_hours"], arms_race_df["voice_plus_face"], label="Voice + face")
plt.plot(arms_race_df["training_hours"], arms_race_df["multimodal"], label="Multimodal")
plt.plot(arms_race_df["training_hours"], arms_race_df["multimodal_plus_coupling"], label="Multimodal + coupling")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Imitator voice training exposure (hours)")
plt.ylabel("Mismatch score (lower is better)")
plt.title("Experiment 12: Imitator improvement vs verification expansion")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()

# Summary table

This cell collects a compact set of headline outputs from the experiments.

In [ ]:
summary = pd.DataFrame([
    {
        "experiment": "1 Voice duration",
        "metric": "Best true rank",
        "value": int(voice_duration_df["true_rank"].min()),
    },
    {
        "experiment": "3 Breadth vs depth",
        "metric": "Median multimodal rank",
        "value": float(breadth_depth_df["multimodal_rank"].median()),
    },
    {
        "experiment": "4 Admissible contraction",
        "metric": "Final admissible candidates",
        "value": int(contraction_df["admissible_candidates"].iloc[-1]),
    },
    {
        "experiment": "5 Cross-modal coupling",
        "metric": "Median rank with coupling",
        "value": float(coupling_df["rank_with_coupling"].median()),
    },
    {
        "experiment": "11 Staleness",
        "metric": "Rank with temporal model",
        "value": int(staleness_df["rank_with_temporal_model"].iloc[0]),
    },
])

summary

# Interpretation

The notebook treats identity as an increasingly constrained hypothesis rather than a binary token.

Three distinct axes matter:

**Depth** means more evidence inside one modality. This usually improves discrimination, but eventually saturates.

**Breadth** means adding modalities that constrain different latent properties. Breadth can shrink the admissible identity set more efficiently than repeatedly measuring an already well-characterized channel.

**Coupling** means testing whether modalities relate to one another in the way expected of one continuing individual. These relationships can reject simulations that are convincing when each modality is judged independently.

Temporal continuity adds another layer: the verifier can ask whether current observations are a plausible continuation of the same historical trajectory, not merely whether they resemble an old snapshot.

The synthetic results should not be interpreted as real biometric performance estimates. Their purpose is to expose the structure of the problem and provide a framework for more careful theoretical work.

# Export

The following cell writes the main experiment tables to CSV files for later analysis.

In [ ]:
OUTPUT_DIR = Path("progressive_identity_experiments")
OUTPUT_DIR.mkdir(exist_ok=True)

tables = {
    "voice_duration": voice_duration_df,
    "repeated_vs_single": repeat_df,
    "breadth_vs_depth": breadth_depth_df,
    "admissible_contraction": contraction_df,
    "cross_modal_coupling": coupling_df,
    "imitation": imitation_df,
    "temporal_continuity": temporal_df,
    "budget_allocation": budget_df,
    "correlated_evidence": correlation_df,
    "conflict_breakdown": conflict_breakdown_df,
    "staleness": staleness_df,
    "arms_race": arms_race_df,
}

for name, df in tables.items():
    path = OUTPUT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)

print(f"Wrote {len(tables)} CSV files to {OUTPUT_DIR.resolve()}")